 ## Data Exploration & Cleaning

In [7]:
import pandas as pd
#loading initial data
df = pd.read_csv("project_dataset.csv", sep=";")

df.shape
df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12070 entries, 0 to 12069
Data columns (total 26 columns):
 #   Column                                                                                       Non-Null Count  Dtype 
---  ------                                                                                       --------------  ----- 
 0   Date                                                                                         12010 non-null  object
 1   Service                                                                                      11830 non-null  object
 2   Gare de départ                                                                               12011 non-null  object
 3   Gare d'arrivée                                                                               12011 non-null  object
 4   Durée moyenne du trajet                                                                      11830 non-null  object
 5   Nombre de circulations prévues         

,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,...,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
count,12010,11830,12011,12011,11830,11830,11831,577,11830,11831,...,11828,11831,11832,11833,11828,11827,11829,11827,11828,11830
unique,318,2,132,116,651,1034,248,3,629,11162,...,265,11075,175,96,2732,2592,2677,2535,2130,2172
top,2025-09,National,PARIS LYON,PARIS LYON,96.0,31.0,0.0,Non communiqué,13.0,0.0,...,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
freq,135,10370,2405,2413,184,117,3917,196,161,134,...,327,218,779,1760,1100,1021,1326,1220,3361,3809


In [8]:
#identifying columns where there are empty values and in what percentage ratio
missing = df.isna().mean().sort_values(ascending=False)*100
missing
df.isna().sum().sort_values(ascending=False)

Commentaire annulations                                                                        11493
Commentaire retards au départ                                                                  11466
Commentaire retards à l'arrivée                                                                10930
Prct retard pour cause matériel roulant                                                          243
Prct retard pour cause infrastructure                                                            243
Prct retard pour causes externes                                                                 242
Retard moyen des trains en retard à l'arrivée                                                    242
Nombre trains en retard > 15min                                                                  242
Prct retard pour cause gestion en gare et réutilisation de matériel                              242
Prct retard pour cause gestion trafic                                                      

In [9]:
#removing almost empty columns
df = df.drop(columns=[
    "Commentaire annulations",
    "Commentaire retards au départ",
    "Commentaire retards à l'arrivée"
])
##convert date type
df["Date"] = pd.to_datetime(df["Date"], errors="coerce", format="mixed")
df.isna().sum().sort_values(ascending=False)

Prct retard pour cause matériel roulant                                                        243
Prct retard pour cause infrastructure                                                          243
Nombre trains en retard > 15min                                                                242
Retard moyen des trains en retard à l'arrivée                                                  242
Prct retard pour causes externes                                                               242
Prct retard pour cause gestion en gare et réutilisation de matériel                            242
Nombre de trains en retard à l'arrivée                                                         241
Prct retard pour cause gestion trafic                                                          241
Retard moyen de tous les trains au départ                                                      241
Nombre de trains en retard au départ                                                           240
Prct retar

In [10]:
#removing lines where at least one critical value is missing
df = df.dropna(subset=[
    "Retard moyen de tous les trains à l'arrivée",
    "Date",
    "Gare de départ",
    "Gare d'arrivée"
])
#check how many are empty

df.isna().sum().sort_values(ascending=False)

Prct retard pour causes externes                                                               238
Nombre trains en retard > 15min                                                                236
Prct retard pour cause gestion trafic                                                          236
Prct retard pour cause matériel roulant                                                        236
Retard moyen des trains en retard à l'arrivée                                                  235
Retard moyen des trains en retard au départ                                                    235
Nombre de trains en retard à l'arrivée                                                         234
Retard moyen de tous les trains au départ                                                      234
Prct retard pour cause gestion en gare et réutilisation de matériel                            233
Durée moyenne du trajet                                                                        233
Nombre tra

In [11]:
text_cols = ["Date", "Service", "Gare de départ", "Gare d'arrivée"]

##standardize all numerical values to a single format, convert numerical columns to float
for col in df.columns:
    if col not in text_cols:
        df[col]=(df[col].astype(str).str.extract(r'(-?\d+[.,]?\d*)')[0].str.replace(",",".", regex=False))
        df[col]=pd.to_numeric(df[col], errors="coerce")

df.isna().sum().sort_values(ascending=False)

Prct retard pour causes externes                                                               238
Nombre trains en retard > 15min                                                                236
Prct retard pour cause gestion trafic                                                          236
Prct retard pour cause matériel roulant                                                        236
Retard moyen des trains en retard à l'arrivée                                                  235
Retard moyen des trains en retard au départ                                                    235
Nombre de trains en retard à l'arrivée                                                         234
Retard moyen de tous les trains au départ                                                      234
Prct retard pour cause gestion en gare et réutilisation de matériel                            233
Durée moyenne du trajet                                                                        233
Nombre tra

In [ ]:
##select all columns with type float64
num_col = df.select_dtypes(include=["float64"]).columns

##filling empty values with column mean values
for col in num_col:
    df[col] = df[col].fillna(df[col].mean())

##Filling empty rows in the "Service" column
df["Service"]=df["Service"].fillna("Unknown")


df = df.drop_duplicates()

float_cols = df.select_dtypes(include=["float64"]).columns

for col in float_cols:
    df = df[df[col] >= 0]

df = df[
    (df["Nombre trains en retard > 30min"] <= df["Nombre trains en retard > 15min"]) &
    (df["Nombre trains en retard > 60min"] <= df["Nombre trains en retard > 30min"])
]


df.to_csv('cleaned_dataset.csv', index=False)

Date                                                                                           0
Service                                                                                        0
Gare de départ                                                                                 0
Gare d'arrivée                                                                                 0
Durée moyenne du trajet                                                                        0
Nombre de circulations prévues                                                                 0
Nombre de trains annulés                                                                       0
Nombre de trains en retard au départ                                                           0
Retard moyen des trains en retard au départ                                                    0
Retard moyen de tous les trains au départ                                                      0
Nombre de trains en retard à l